In [42]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql import functions as F


StatementMeta(, 08b431bc-b63f-4868-bda7-50c777c1d7bd, 44, Finished, Available, Finished, False)

In [43]:
silver_table_path="abfss://88d42a16-2c0a-495a-97db-34b2c41a2439@onelake.dfs.fabric.microsoft.com/fd8da432-bd8a-4833-8419-d9bde4a5ee74/Tables/dbo/produtos"

df = spark.read.format("delta").load(silver_table_path)

display(df)

StatementMeta(, 08b431bc-b63f-4868-bda7-50c777c1d7bd, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f25a66da-48b2-47e0-a874-8f533c8108af)

In [44]:
date_dim = df.select("date", "dia", "mes", "quarter", "ano").distinct() \
                .withColumnRenamed("date", "date_id")

StatementMeta(, 08b431bc-b63f-4868-bda7-50c777c1d7bd, 46, Finished, Available, Finished, False)

In [45]:
time_dim = df.select("time", "hora", "minuto",  "periodo").distinct() \
                .withColumnRenamed("time", "time_id")

StatementMeta(, 08b431bc-b63f-4868-bda7-50c777c1d7bd, 47, Finished, Available, Finished, False)

In [46]:
# Create the Turbine Dimension Table
categoria_dim = df.select("categoria").distinct() \
                .withColumn("categoria_id", row_number().over(Window.orderBy("categoria")))


StatementMeta(, 08b431bc-b63f-4868-bda7-50c777c1d7bd, 48, Finished, Available, Finished, False)

In [47]:
# Create the Turbine Dimension Table
departamento_dim = df.select("departamento").distinct() \
                .withColumn("departamento_id", row_number().over(Window.orderBy("departamento")))


StatementMeta(, 08b431bc-b63f-4868-bda7-50c777c1d7bd, 49, Finished, Available, Finished, False)

In [48]:
produto_fact = (
    df.select("produto_id", "nome_produto", "status","valor", "date", "time" ,"categoria", "departamento")
      .dropDuplicates(["produto_id"])
      .join(categoria_dim, on="categoria", how="left")
      .join(departamento_dim, on="departamento", how="left")  
      .select(
          "produto_id",
          F.col("date").alias("date_id"),
          F.col("time").alias("time_id"),
          "nome_produto",
          "valor",
          "status",
          "categoria_id",
          "departamento_id"
      )
)


StatementMeta(, 08b431bc-b63f-4868-bda7-50c777c1d7bd, 50, Finished, Available, Finished, False)

In [49]:
# Paths to the Gold tables
gold_date_dim_path = "abfss://88d42a16-2c0a-495a-97db-34b2c41a2439@onelake.dfs.fabric.microsoft.com/d557d27d-a995-487d-9b0f-c5561b8be104/Tables/dbo/dim_date"
gold_time_dim_path = "abfss://88d42a16-2c0a-495a-97db-34b2c41a2439@onelake.dfs.fabric.microsoft.com/d557d27d-a995-487d-9b0f-c5561b8be104/Tables/dbo/dim_time"
gold_categoria_dim_path = "abfss://88d42a16-2c0a-495a-97db-34b2c41a2439@onelake.dfs.fabric.microsoft.com/d557d27d-a995-487d-9b0f-c5561b8be104/Tables/dbo/dim_categoria"
gold_departamento_dim_path = "abfss://88d42a16-2c0a-495a-97db-34b2c41a2439@onelake.dfs.fabric.microsoft.com/d557d27d-a995-487d-9b0f-c5561b8be104/Tables/dbo/departamento_dim"
gold_fact_table_path = "abfss://88d42a16-2c0a-495a-97db-34b2c41a2439@onelake.dfs.fabric.microsoft.com/d557d27d-a995-487d-9b0f-c5561b8be104/Tables/dbo/produto_fact"

StatementMeta(, 08b431bc-b63f-4868-bda7-50c777c1d7bd, 51, Finished, Available, Finished, False)

In [50]:
date_dim.write.format("delta").mode("overwrite").save(gold_date_dim_path)
time_dim.write.format("delta").mode("overwrite").save(gold_time_dim_path)
categoria_dim.write.format("delta").mode("overwrite").save(gold_categoria_dim_path)
departamento_dim.write.format("delta").mode("overwrite").save(gold_departamento_dim_path)
produto_fact.write.format("delta").mode("overwrite").save(gold_fact_table_path)

StatementMeta(, 08b431bc-b63f-4868-bda7-50c777c1d7bd, 52, Finished, Available, Finished, False)